In [1]:
pip install anthropic pillow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install pandas openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import base64
import json
import os
import re

import numpy as np
import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

COLUMNS_ORDER = [
    "Giorno", "Gen", "Feb", "Mar", "Apr", "Mag", "Giu", "Lug", "Ago", "Set", "Ott", "Nov", "Dic"
]


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def normalize_table_cell_to_number(v):
    """Ground-truth style: blanks / noise → 0; comma decimals; strip brackets (do not treat [ as a digit)."""
    if v is None:
        return 0.0
    if isinstance(v, (int, np.integer)):
        return float(v)
    if isinstance(v, (float, np.floating)):
        return 0.0 if pd.isna(v) else float(v)
    s = str(v).strip()
    if s == "" or s.lower() in {"null", "none", "nan", "-", "—", "–", ".", ".."}:
        return 0.0
    s = s.replace(",", ".")
    s = re.sub(r"^\[+|\]+$", "", s)
    s = s.strip("()[] ")
    s = s.replace("*", "").strip()
    if s == "" or s == ".":
        return 0.0
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    if not m:
        return 0.0
    try:
        return float(m.group(0))
    except ValueError:
        return 0.0


def extract_json_array(raw_text: str):
    m = re.search(r"\[\s*\{.*\}\s*\]", raw_text, re.DOTALL)
    if m:
        return m.group(0)
    m = re.search(r"\[.*\]", raw_text, re.DOTALL)
    return m.group(0) if m else None


def validate_parsed_wide_df(df: pd.DataFrame):
    if df is None or df.empty:
        return False, "empty"
    if len(df) != 31:
        return False, f"expected 31 rows, got {len(df)}"
    if "Giorno" not in df.columns:
        return False, "missing Giorno"
    g = pd.to_numeric(df["Giorno"], errors="coerce")
    if g.isna().any():
        return False, "Giorno NaN"
    days = sorted(int(x) for x in g.tolist())
    if days != list(range(1, 32)):
        return False, f"Giorno not 1..31: {days[:8]}..."
    return True, "ok"


def wide_df_from_json(json_data: list) -> pd.DataFrame:
    df = pd.DataFrame(json_data)
    for col in COLUMNS_ORDER:
        if col not in df.columns:
            df[col] = np.nan
    df = df[COLUMNS_ORDER].copy()
    df["Giorno"] = pd.to_numeric(df["Giorno"], errors="coerce").astype(int)
    for col in COLUMNS_ORDER[1:]:
        df[col] = [normalize_table_cell_to_number(x) for x in df[col].tolist()]
    return df.sort_values("Giorno").reset_index(drop=True)


def extract_table_data(image_path, image_media_type="image/gif", extra_user_text=""):
    image_data = encode_image(image_path)

    system_instruction = """You are an elite data auditor for Italian hydrological yearbook discharge tables.

GOAL
Transcribe the 31×12 daily discharge grid with zero structural mistakes.

LAYOUT
- First column: day index 1..31 (Giorno / G. / unlabeled).
- Next 12 columns: Jan through Dec (Italian labels may abbreviate or use roman numerals).
- Never copy footer rows (Media, Massima, Minima, Totale, Totali).

CRITICAL READING RULES
1) BRACKETS: Values like [12.5] or [0.65] use brackets as punctuation ONLY. Never read '[' as '1','3', or '6'. A common failure is [0.65] misread as 6.65—extract only the number inside.
2) DECIMALS: Commas may be decimal separators (12,34 → 12.34).
3) EMPTY CELLS: Blank, '.', or dashes mean 0. Output JSON number 0, not null.
4) ALIGNMENT: Italic or compressed months (often mid-year onward)—anchor every cell to the printed day row; day 15 must align across all months.

QUALITY (MANDATORY)
- Vertical pass: per month, count filled rows in the 31-day block; must match month length (28/29/30/31).
- Horizontal pass: same row index for each day across all months.

DIGITS
- 2 vs 9 (base of 2 is flatter); 3 vs 8 (loops); 6 vs 0 in small values like 0.xx.

OUTPUT
- ONLY a JSON array of exactly 31 objects, no markdown or prose.
- Keys in each object, this order only: "Giorno","Gen","Feb","Mar","Apr","Mag","Giu","Lug","Ago","Set","Ott","Nov","Dic".
- Giorno: integers 1..31 once each, in order.
- All month values are JSON numbers (0 if missing)."""

    user_instruction = (
        (extra_user_text.strip() + "\n\n") if extra_user_text else ""
    ) + (
        "Extract the full table. Brackets are not digits. Return ONLY the JSON array."
    )

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=16384,
        temperature=0,
        system=system_instruction,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": image_media_type,
                            "data": image_data,
                        },
                    },
                    {"type": "text", "text": user_instruction},
                ],
            }
        ],
    )
    return message.content[0].text


def parse_and_validate_response(raw_output: str):
    json_str = extract_json_array(raw_output)
    if not json_str:
        return None, "no_json"
    df = wide_df_from_json(json.loads(json_str))
    ok, reason = validate_parsed_wide_df(df)
    if not ok:
        return None, reason
    return df, "ok"


# --- Batch extraction ---
input_folder = "../test_pictures/"
columns_order = COLUMNS_ORDER

if not os.path.exists(input_folder):
    print(f"Error: Folder {input_folder} not found.")
else:
    files = sorted(f for f in os.listdir(input_folder) if f.lower().endswith(".gif"))
    print(f"Found {len(files)} GIF benchmark(s). Starting extraction...")

    for filename in files:
        image_path = os.path.join(input_folder, filename)
        base_name = os.path.splitext(filename)[0]
        output_filename = f"{base_name}_claude.xlsx"

        print(f"Processing: {filename}...")

        try:
            raw_output = extract_table_data(image_path)
            df, status = parse_and_validate_response(raw_output)

            if df is None:
                print(f"⚠️ First pass failed ({status}) — retrying...")
                raw_output = extract_table_data(
                    image_path,
                    extra_user_text=(
                        f"CRITICAL: Prior output invalid ({status}). "
                        "Return EXACTLY 31 rows, Giorno 1..31, perfect row alignment."
                    ),
                )
                df, status = parse_and_validate_response(raw_output)

            if df is None:
                print(f"❌ Giving up after retry ({status}): {filename}")
                continue

            df.to_excel(output_filename, index=False)
            print(f"✅ Saved: {output_filename}")

        except Exception as e:
            print(f"❌ Error in {filename}: {e}")

print("\nAll files processed successfully.")

Found 15 GIF benchmark(s). Starting extraction...
Processing: Q_Bari_1969_26.gif...
✅ Saved: Q_Bari_1969_26_claude.xlsx
Processing: Q_Cagliari_1924_66.gif...
✅ Saved: Q_Cagliari_1924_66_claude.xlsx
Processing: Q_Cagliari_1933_91.gif...
✅ Saved: Q_Cagliari_1933_91_claude.xlsx
Processing: Q_Catanzaro_1937_36.gif...
✅ Saved: Q_Catanzaro_1937_36_claude.xlsx
Processing: Q_Catanzaro_1940_54.gif...
✅ Saved: Q_Catanzaro_1940_54_claude.xlsx
Processing: Q_Genova_1936_86.gif...
✅ Saved: Q_Genova_1936_86_claude.xlsx
Processing: Q_Parma_1942_75.gif...
✅ Saved: Q_Parma_1942_75_claude.xlsx
Processing: Q_Pescara_1926_65.gif...
✅ Saved: Q_Pescara_1926_65_claude.xlsx
Processing: Q_Pescara_1947_73.gif...
✅ Saved: Q_Pescara_1947_73_claude.xlsx
Processing: Q_Pisa_1935_66.gif...
✅ Saved: Q_Pisa_1935_66_claude.xlsx
Processing: Q_Roma_1927_79.gif...
✅ Saved: Q_Roma_1927_79_claude.xlsx
Processing: Q_Roma_1928_73.gif...
✅ Saved: Q_Roma_1928_73_claude.xlsx
Processing: Q_Roma_1980_30.gif...
❌ Error in Q_Roma_1980

In [4]:
import os
import re

import numpy as np
import pandas as pd

input_folder = "./"
output_folder = "./reformatted_results/"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

month_map = {
    "Gen": 1, "Feb": 2, "Mar": 3, "Apr": 4, "Mag": 5, "Giu": 6,
    "Lug": 7, "Ago": 8, "Set": 9, "Ott": 10, "Nov": 11, "Dic": 12,
}


def _coerce_day(v) -> int:
    x = pd.to_numeric(v, errors="coerce")
    if pd.isna(x):
        return 0
    return int(round(float(x)))


def _coerce_value(v) -> float:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return 0.0
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)
    s = str(v).strip().replace(",", ".")
    if s == "" or s.lower() in {"null", "none", "-", ".", ".."}:
        return 0.0
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group(0)) if m else 0.0


files = sorted(f for f in os.listdir(input_folder) if f.endswith("_claude.xlsx"))

print(f"Found {len(files)} files to reformat.")

for filename in files:
    file_path = os.path.join(input_folder, filename)

    try:
        df_claude = pd.read_excel(file_path)

        year_match = re.search(r"\d{4}", filename)
        current_year = int(year_match.group(0)) if year_match else 0

        df_claude["Giorno"] = df_claude["Giorno"].map(_coerce_day)

        df_melted = df_claude.melt(id_vars=["Giorno"], var_name="Month_Name", value_name="Value")
        df_melted["Value"] = df_melted["Value"].map(_coerce_value)

        df_final = pd.DataFrame()
        df_final[0] = [current_year] * len(df_melted)
        df_final[1] = df_melted["Month_Name"].map(month_map).astype(int)
        df_final[2] = df_melted["Giorno"].astype(int)
        df_final[3] = df_melted["Value"].astype(float)

        df_final = df_final.sort_values(by=[1, 2])

        base_name = filename.replace("_claude.xlsx", "")
        output_name = f"{base_name}_reform.xlsx"

        df_final.to_excel(os.path.join(output_folder, output_name), index=False, header=False)

        print(f"✅ Reformat Done: {output_name}")

    except Exception as e:
        print(f"❌ Error in {filename}: {e}")

print(f"\nReformat complete ({len(files)} files).")

Found 15 files to reformat.
✅ Reformat Done: Q_Bari_1969_26_reform.xlsx
✅ Reformat Done: Q_Cagliari_1924_66_reform.xlsx
✅ Reformat Done: Q_Cagliari_1933_91_reform.xlsx
✅ Reformat Done: Q_Catanzaro_1937_36_reform.xlsx
✅ Reformat Done: Q_Catanzaro_1940_54_reform.xlsx
✅ Reformat Done: Q_Genova_1936_86_reform.xlsx
✅ Reformat Done: Q_Parma_1942_75_reform.xlsx
✅ Reformat Done: Q_Pescara_1926_65_reform.xlsx
✅ Reformat Done: Q_Pescara_1947_73_reform.xlsx
✅ Reformat Done: Q_Pisa_1935_66_reform.xlsx
✅ Reformat Done: Q_Roma_1927_79_reform.xlsx
✅ Reformat Done: Q_Roma_1928_73_reform.xlsx
✅ Reformat Done: Q_Roma_1980_30_reform.xlsx
✅ Reformat Done: Q_Venezia_1926_138_reform.xlsx
✅ Reformat Done: Q_Venezia_1935_186_reform.xlsx

Reformat complete (15 files).


In [5]:
import pandas as pd
import numpy as np
import os
import re

# ۱. تنظیمات پوشه‌ها
reformatted_folder = "./reformatted_results/" 
manual_folder = "../test_pictures/"          

reform_files = [f for f in os.listdir(reformatted_folder) if f.endswith('_reform.xlsx')]
all_accuracies = []

print(f"--- شروع مقایسه با اصلاح منطق NaN/Zero برای {len(reform_files)} فایل ---\n")

for ref_file in reform_files:
    base_name = ref_file.replace('_reform.xlsx', '')
    manual_file = f"{base_name}.xlsx"
    
    ref_path = os.path.join(reformatted_folder, ref_file)
    man_path = os.path.join(manual_folder, manual_file)
    
    if not os.path.exists(man_path):
        continue

    try:
        # ۲. بارگذاری فایل‌ها
        df_manual = pd.read_excel(man_path, header=None)
        df_reform = pd.read_excel(ref_path, header=None)

        # ۳. انتخاب ستون‌های ۱ تا ۳ (ماه، روز، مقدار)
        df_man_3cols = df_manual.iloc[:, 1:4].copy()
        df_ref_3cols = df_reform.iloc[:, 1:4].copy()

        df_man_3cols.columns = ['Month', 'Day', 'Value_Man']
        df_ref_3cols.columns = ['Month', 'Day', 'Value_Ref']

        # ۴. یکسان‌سازی مقادیر
        df_man_3cols['Value_Man'] = pd.to_numeric(df_man_3cols['Value_Man'], errors='coerce').fillna(0)
        df_ref_3cols['Value_Ref'] = pd.to_numeric(df_ref_3cols['Value_Ref'], errors='coerce').fillna(0)

        df_man_3cols['Month'] = pd.to_numeric(df_man_3cols['Month'], errors='coerce').fillna(-1).astype(int)
        df_man_3cols['Day'] = pd.to_numeric(df_man_3cols['Day'], errors='coerce').fillna(-1).astype(int)
        df_ref_3cols['Month'] = pd.to_numeric(df_ref_3cols['Month'], errors='coerce').fillna(-1).astype(int)
        df_ref_3cols['Day'] = pd.to_numeric(df_ref_3cols['Day'], errors='coerce').fillna(-1).astype(int)

        # ۵. فیلتر سال (ستون سال در XLSX دستی اغلب float است، مثلاً 1933.0)
        year_match = re.search(r'\d{4}', base_name)
        if year_match:
            target_year = int(year_match.group(0))
            year_series = pd.to_numeric(df_manual.iloc[:, 0], errors='coerce')
            df_man_final = df_man_3cols[np.isclose(year_series, float(target_year), rtol=0, atol=0.001)].copy()
        else:
            df_man_final = df_man_3cols.copy()

        comparison = pd.merge(df_man_final, df_ref_3cols, on=['Month', 'Day'], how='inner')

        comparison['Is_Match'] = np.isclose(
            comparison['Value_Man'].astype(float),
            comparison['Value_Ref'].astype(float),
            atol=1e-3,
        )

        total_points = len(comparison)
        matches = comparison['Is_Match'].sum()
        accuracy = (matches / total_points) * 100 if total_points > 0 else 0
        mismatch_count = int(total_points - matches)
        target_ok = " 🎯" if mismatch_count <= 2 else ""

        print(f"✅ {base_name}: Accuracy = {accuracy:.2f}% ({mismatch_count} errors){target_ok}")

        all_accuracies.append({
            'Table': base_name,
            'Total_Days': total_points,
            'Matches': int(matches),
            'Errors': mismatch_count,
            'Accuracy': accuracy,
        })

    except Exception as e:
        print(f"❌ Error in {base_name}: {e}")

# ۸. خروجی نهایی
summary_df = pd.DataFrame(all_accuracies)
summary_df.to_excel("final_accuracy_batch_report_v2.xlsx", index=False)
print("\nگزارش نهایی در فایل 'final_accuracy_batch_report_v2.xlsx' ذخیره شد.")

--- شروع مقایسه با اصلاح منطق NaN/Zero برای 15 فایل ---

✅ Q_Cagliari_1933_91: Accuracy = 94.52% (20 errors)
✅ Q_Venezia_1935_186: Accuracy = 92.33% (28 errors)
✅ Q_Cagliari_1924_66: Accuracy = 93.72% (23 errors)
✅ Q_Catanzaro_1940_54: Accuracy = 77.05% (84 errors)
✅ Q_Bari_1969_26: Accuracy = 100.00% (0 errors) 🎯
✅ Q_Pescara_1926_65: Accuracy = 98.63% (5 errors)
✅ Q_Pescara_1947_73: Accuracy = 96.71% (12 errors)
✅ Q_Genova_1936_86: Accuracy = 77.87% (81 errors)
✅ Q_Venezia_1926_138: Accuracy = 96.44% (13 errors)
✅ Q_Parma_1942_75: Accuracy = 89.32% (39 errors)
✅ Q_Catanzaro_1937_36: Accuracy = 86.85% (48 errors)
✅ Q_Roma_1927_79: Accuracy = 93.70% (23 errors)
✅ Q_Roma_1980_30: Accuracy = 98.63% (5 errors)
✅ Q_Pisa_1935_66: Accuracy = 90.14% (36 errors)
✅ Q_Roma_1928_73: Accuracy = 95.90% (15 errors)

گزارش نهایی در فایل 'final_accuracy_batch_report_v2.xlsx' ذخیره شد.


In [6]:
import pandas as pd
import numpy as np
import os
import re

# ۱. تنظیمات پوشه‌ها
reformatted_folder = "./reformatted_results/" 
manual_folder = "../test_pictures/"          
debug_folder = "./debug_mismatches/" # پوشه جدید برای بررسی خطاها

if not os.path.exists(debug_folder):
    os.makedirs(debug_folder)

reform_files = [f for f in os.listdir(reformatted_folder) if f.endswith('_reform.xlsx')]
all_accuracies = []

print(f"--- شروع مقایسه و عیب‌یابی برای {len(reform_files)} فایل ---\n")

for ref_file in reform_files:
    base_name = ref_file.replace('_reform.xlsx', '')
    manual_file = f"{base_name}.xlsx"
    
    ref_path = os.path.join(reformatted_folder, ref_file)
    man_path = os.path.join(manual_folder, manual_file)
    
    if not os.path.exists(man_path):
        continue

    try:
        df_manual = pd.read_excel(man_path, header=None)
        df_reform = pd.read_excel(ref_path, header=None)

        # انتخاب ستون‌های اصلی
        df_man_3cols = df_manual.iloc[:, 1:4].copy()
        df_ref_3cols = df_reform.iloc[:, 1:4].copy()
        df_man_3cols.columns = ['Month', 'Day', 'Value_Man']
        df_ref_3cols.columns = ['Month', 'Day', 'Value_Ref']

        df_man_3cols['Value_Man'] = pd.to_numeric(df_man_3cols['Value_Man'], errors='coerce').fillna(0)
        df_ref_3cols['Value_Ref'] = pd.to_numeric(df_ref_3cols['Value_Ref'], errors='coerce').fillna(0)

        df_man_3cols['Month'] = pd.to_numeric(df_man_3cols['Month'], errors='coerce').fillna(-1).astype(int)
        df_man_3cols['Day'] = pd.to_numeric(df_man_3cols['Day'], errors='coerce').fillna(-1).astype(int)
        df_ref_3cols['Month'] = pd.to_numeric(df_ref_3cols['Month'], errors='coerce').fillna(-1).astype(int)
        df_ref_3cols['Day'] = pd.to_numeric(df_ref_3cols['Day'], errors='coerce').fillna(-1).astype(int)

        year_match = re.search(r'\d{4}', base_name)
        target_year = int(year_match.group(0)) if year_match else None
        if target_year:
            year_series = pd.to_numeric(df_manual.iloc[:, 0], errors='coerce')
            df_man_final = df_man_3cols[np.isclose(year_series, float(target_year), rtol=0, atol=0.001)].copy()
        else:
            df_man_final = df_man_3cols.copy()

        comparison = pd.merge(df_man_final, df_ref_3cols, on=['Month', 'Day'], how='inner')

        comparison['Is_Match'] = np.isclose(
            comparison['Value_Man'].astype(float),
            comparison['Value_Ref'].astype(float),
            atol=1e-3,
        )

        total_points = len(comparison)
        matches = comparison['Is_Match'].sum()
        accuracy = (matches / total_points) * 100 if total_points > 0 else 0
        mismatch_count = int(total_points - matches)

        if mismatch_count > 2:
            mismatches = comparison[comparison['Is_Match'] == False].copy()
            if not mismatches.empty:
                mismatches['Diff'] = mismatches['Value_Man'] - mismatches['Value_Ref']
                error_file = os.path.join(debug_folder, f"mismatches_{base_name}.xlsx")
                mismatches.to_excel(error_file, index=False)
                print(f"⚠️ {base_name}: {accuracy:.2f}% ({mismatch_count} errors) → {error_file}")
            else:
                print(f"✅ {base_name}: {accuracy:.2f}%")
        else:
            print(f"✅ {base_name}: {accuracy:.2f}% ({mismatch_count} errors) 🎯")

        all_accuracies.append({
            'Table': base_name,
            'Accuracy': accuracy,
            'Errors': mismatch_count,
            'Total_Days': int(total_points),
        })

    except Exception as e:
        print(f"❌ Error in {base_name}: {e}")

# گزارش نهایی
summary_df = pd.DataFrame(all_accuracies)
summary_df.to_excel("final_batch_accuracy_with_debug.xlsx", index=False)
print("\nفرآیند تمام شد. برای بررسی دلایل افت دقت، پوشه 'debug_mismatches' رو چک کن.")

--- شروع مقایسه و عیب‌یابی برای 15 فایل ---

⚠️ Q_Cagliari_1933_91: 94.52% (20 errors) → ./debug_mismatches/mismatches_Q_Cagliari_1933_91.xlsx
⚠️ Q_Venezia_1935_186: 92.33% (28 errors) → ./debug_mismatches/mismatches_Q_Venezia_1935_186.xlsx
⚠️ Q_Cagliari_1924_66: 93.72% (23 errors) → ./debug_mismatches/mismatches_Q_Cagliari_1924_66.xlsx
⚠️ Q_Catanzaro_1940_54: 77.05% (84 errors) → ./debug_mismatches/mismatches_Q_Catanzaro_1940_54.xlsx
✅ Q_Bari_1969_26: 100.00% (0 errors) 🎯
⚠️ Q_Pescara_1926_65: 98.63% (5 errors) → ./debug_mismatches/mismatches_Q_Pescara_1926_65.xlsx
⚠️ Q_Pescara_1947_73: 96.71% (12 errors) → ./debug_mismatches/mismatches_Q_Pescara_1947_73.xlsx
⚠️ Q_Genova_1936_86: 77.87% (81 errors) → ./debug_mismatches/mismatches_Q_Genova_1936_86.xlsx
⚠️ Q_Venezia_1926_138: 96.44% (13 errors) → ./debug_mismatches/mismatches_Q_Venezia_1926_138.xlsx
⚠️ Q_Parma_1942_75: 89.32% (39 errors) → ./debug_mismatches/mismatches_Q_Parma_1942_75.xlsx
⚠️ Q_Catanzaro_1937_36: 86.85% (48 errors) → ./

In [7]:
# --- Optional: guided second pass (benchmark workflow) ---
# Uses ONLY (month, day) positions from mismatch files—never sends correct discharge values to the API.
# Prefers ../claude_chain2/debug_mismatches if that folder exists (shared with main chain2 runs).

import os
import re

import numpy as np
import pandas as pd

ENABLE_GUIDED_REEXTRACTION = True
MAX_HINT_PAIRS = 48
GIF_FOLDER = "../test_pictures/"
MANUAL_FOLDER = "../test_pictures/"
LOCAL_DEBUG = "./debug_mismatches"
SHARED_DEBUG = "../claude_chain2/debug_mismatches"
debug_source = SHARED_DEBUG if os.path.isdir(SHARED_DEBUG) else LOCAL_DEBUG

MONTH_NUM_TO_KEY = {m: COLUMNS_ORDER[m] for m in range(1, 13)}


def count_errors_vs_manual(base_name: str, df_wide: pd.DataFrame) -> int:
    """Errors vs ground-truth XLSX (same rules as accuracy cell)."""
    month_map = {
        "Gen": 1, "Feb": 2, "Mar": 3, "Apr": 4, "Mag": 5, "Giu": 6,
        "Lug": 7, "Ago": 8, "Set": 9, "Ott": 10, "Nov": 11, "Dic": 12,
    }
    man_path = os.path.join(MANUAL_FOLDER, f"{base_name}.xlsx")
    if not os.path.exists(man_path):
        return 10**9

    df_manual = pd.read_excel(man_path, header=None)
    df_man_3 = df_manual.iloc[:, 1:4].copy()
    df_man_3.columns = ["Month", "Day", "Value_Man"]
    df_man_3["Value_Man"] = pd.to_numeric(df_man_3["Value_Man"], errors="coerce").fillna(0)
    df_man_3["Month"] = pd.to_numeric(df_man_3["Month"], errors="coerce").fillna(-1).astype(int)
    df_man_3["Day"] = pd.to_numeric(df_man_3["Day"], errors="coerce").fillna(-1).astype(int)

    ym = re.search(r"\d{4}", base_name)
    target_year = int(ym.group(0)) if ym else None
    if target_year:
        ycol = pd.to_numeric(df_manual.iloc[:, 0], errors="coerce")
        df_man_f = df_man_3[np.isclose(ycol, float(target_year), rtol=0, atol=0.001)].copy()
    else:
        df_man_f = df_man_3

    dc = df_wide.copy()
    dc["Giorno"] = pd.to_numeric(dc["Giorno"], errors="coerce").fillna(0).astype(int)
    for c in COLUMNS_ORDER[1:]:
        dc[c] = [normalize_table_cell_to_number(v) for v in dc[c].tolist()]
    melted = dc.melt(id_vars=["Giorno"], var_name="Month_Name", value_name="Value")
    ref = pd.DataFrame()
    ref["Month"] = melted["Month_Name"].map(month_map).astype(int)
    ref["Day"] = melted["Giorno"].astype(int)
    ref["Value_Ref"] = melted["Value"].astype(float)

    comp = pd.merge(df_man_f, ref, on=["Month", "Day"], how="inner")
    ok = np.isclose(comp["Value_Man"].astype(float), comp["Value_Ref"].astype(float), atol=1e-3)
    return int((~ok).sum())


if ENABLE_GUIDED_REEXTRACTION and os.path.isdir(debug_source):
    mm_files = sorted(
        f for f in os.listdir(debug_source)
        if f.startswith("mismatches_") and f.endswith(".xlsx")
    )
    print(f"Guided re-extraction using {len(mm_files)} mismatch file(s) from {debug_source}\n")

    for mf in mm_files:
        base_name = mf.replace("mismatches_", "").replace(".xlsx", "")
        gif_name = f"{base_name}.gif"
        gif_path = os.path.join(GIF_FOLDER, gif_name)
        out_xlsx = f"{base_name}_claude.xlsx"
        if not os.path.exists(gif_path) or not os.path.exists(out_xlsx):
            print(f"⏭ Skip {base_name} (missing gif or claude xlsx)")
            continue

        mm = pd.read_excel(os.path.join(debug_source, mf))
        if len(mm) <= 2:
            print(f"⏭ {base_name}: already ≤2 errors in snapshot")
            continue

        pairs = mm[["Month", "Day"]].drop_duplicates().head(MAX_HINT_PAIRS)
        labels = []
        for r in pairs.itertuples(index=False):
            mk = MONTH_NUM_TO_KEY.get(int(r.Month), "?")
            labels.append(f"{mk}-{int(r.Day)}")
        hint = (
            "Pay maximum attention to these month-day cells (Italian month key + day): "
            + ", ".join(labels)
            + ". Re-check brackets ( [ ] are NOT digits), tiny decimals like 0.xx, and row alignment."
        )

        try:
            prev = pd.read_excel(out_xlsx)
            err_before = count_errors_vs_manual(base_name, prev)
        except Exception:
            err_before = 10**9

        print(f"↻ {base_name}: snapshot had {len(mm)} mismatches; scoring baseline errors={err_before}")

        try:
            raw = extract_table_data(gif_path, extra_user_text=hint)
            df_new, st = parse_and_validate_response(raw)
            if df_new is None:
                raw = extract_table_data(
                    gif_path,
                    extra_user_text=hint + " STRUCTURE MUST be 31 rows, Giorno 1..31.",
                )
                df_new, st = parse_and_validate_response(raw)
            if df_new is None:
                print(f"❌ {base_name}: refine parse failed ({st})")
                continue

            err_after = count_errors_vs_manual(base_name, df_new)
            if err_after < err_before:
                df_new.to_excel(out_xlsx, index=False)
                print(f"   ✅ Kept NEW extract: errors {err_before} → {err_after}")
            else:
                print(f"   ⏺ Kept PREVIOUS extract: errors {err_before} (new would be {err_after})")
        except Exception as e:
            print(f"❌ {base_name}: {e}")

    print("\nRe-run the reformat + accuracy cells to refresh *_reform.xlsx and reports.")
else:
    print(
        "Set ENABLE_GUIDED_REEXTRACTION=True and run compare/debug first, "
        "or create ../claude_chain2/debug_mismatches with mismatch exports."
    )

Guided re-extraction using 12 mismatch file(s) from ../claude_chain2/debug_mismatches

↻ Q_Cagliari_1924_66: snapshot had 30 mismatches; scoring baseline errors=23
❌ Q_Cagliari_1924_66: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CaDPv2HzMWL8DH1hH2jyQ'}
↻ Q_Cagliari_1933_91: snapshot had 19 mismatches; scoring baseline errors=20
❌ Q_Cagliari_1933_91: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CaDPv4wjfDTHwPorz3KCM'}
↻ Q_Catanzaro_1937_36: snapshot had 61 mismatches; scoring baseline errors=48
❌ Q_Catanzaro_1937_36: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Y